In [1]:
from nichenetpy.prediction import LigandActivityPredictor, LigandReceptorNetwork
from nichenetpy.utils import read_matrix_from_csv
from nichenetpy.extraction import get_expressed_genes, subset_ann_celltype

import anndata
import scanpy as sc

In [2]:
predictor = LigandActivityPredictor(*read_matrix_from_csv("D:/Data/nichenetpy/testargs/ligand_target_matrix.csv"))
lr_network = LigandReceptorNetwork(filename="D:/Data/nichenetpy/csv/lr_network.csv")
ann = anndata.io.read_h5ad("D:/Data/nichenetpy/annData/annData3531889.h5")
ann

AnnData object with n_obs × n_vars = 5027 × 13541
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nGene', 'nUMI', 'aggregate', 'res.0.6', 'celltype'
    var: 'gene'
    layers: 'counts', 'data', 'scale.data'

In [3]:
celltype_counts = ann.obs["celltype"].value_counts()
celltype_counts

celltype
CD4 T    2562
CD8 T    1645
B         382
Treg      199
NK        131
Mono       90
DC         18
Name: count, dtype: int64

In [4]:
receiver = "CD8 T"
expressed_genes_receiver = set(get_expressed_genes(receiver, ann, 0.05))
all_receptors = lr_network.get_receptors()
expressed_receptors = all_receptors.intersection(expressed_genes_receiver)
potential_ligands = set(key for key, group in lr_network.item_iter() if len(group.intersection(expressed_receptors)) > 0)

In [5]:
sender_celltypes = ("CD4 T", "Treg", "Mono", "NK", "B", "DC")
list_expressed_genes_sender = [get_expressed_genes(ct, ann, pct=0.05) for ct in sender_celltypes]
expressed_genes_sender = set(e for l in list_expressed_genes_sender for e in l)
potential_ligands_focused = potential_ligands.intersection(expressed_genes_sender)

In [6]:
ann_receiver = subset_ann_celltype(ann, receiver, layers=["data"])

In [7]:
sc.pp.log1p(ann_receiver, layer="data")
sc.tl.rank_genes_groups(ann_receiver, groupby="aggregate", method="wilcoxon", layer="data")

In [18]:
ann_receiver.uns["rank_genes_groups"]

{'params': {'groupby': 'aggregate',
  'reference': 'rest',
  'method': 'wilcoxon',
  'use_raw': False,
  'layer': 'data',
  'corr_method': 'benjamini-hochberg'},
 'names': rec.array([('5535', '3912'), ('5711', '4910'), ('6161', '10292'), ...,
            ('10292', '6161'), ('4910', '5711'), ('3912', '5535')],
           dtype=[('LCMV', 'O'), ('SS', 'O')]),
 'scores': rec.array([( 26.05892 ,  15.315716), ( 23.839825,  13.467089),
            ( 23.42286 ,  13.245551), ..., (-13.245551, -23.42286 ),
            (-13.467089, -23.839825), (-15.315716, -26.05892 )],
           dtype=[('LCMV', '<f4'), ('SS', '<f4')]),
 'pvals': rec.array([(1.06599979e-149, 6.00470076e-053),
            (1.29124930e-125, 2.44309170e-041),
            (2.50010269e-121, 4.78755922e-040), ...,
            (4.78755922e-040, 2.50010269e-121),
            (2.44309170e-041, 1.29124930e-125),
            (6.00470076e-053, 1.06599979e-149)],
           dtype=[('LCMV', '<f8'), ('SS', '<f8')]),
 'pvals_adj': rec.array([(

In [ ]:
geneset_oi = [
    ann.var["gene"].iloc[int(gene)] for gene, pval_adj, log2FC in
    zip(
        [e[0] for e in ann_receiver.uns["rank_genes_groups"]["names"]],
        [e[0] for e in ann_receiver.uns["rank_genes_groups"]["pvals_adj"]],
        [e[0] for e in ann_receiver.uns["rank_genes_groups"]["logfoldchanges"]]
    ) if pval_adj <= 0.05 and abs(log2FC) >= 0.25
]

270

In [17]:
[e[0] for e in ann_receiver.uns["rank_genes_groups"]["pvals_adj"]]

[np.float64(1.443470310438093e-145),
 np.float64(8.742403405553676e-122),
 np.float64(1.128463016538779e-117),
 np.float64(1.0473489997720037e-108),
 np.float64(9.645808240821693e-88),
 np.float64(1.6747379044060675e-85),
 np.float64(1.2508318097989116e-75),
 np.float64(1.0090500424946569e-69),
 np.float64(1.36387537585333e-67),
 np.float64(2.0318210750588664e-66),
 np.float64(7.225046489574329e-61),
 np.float64(1.5695719039550424e-60),
 np.float64(1.3087112757702547e-59),
 np.float64(7.998030918190996e-58),
 np.float64(2.5879469566672867e-57),
 np.float64(7.138549466330522e-56),
 np.float64(8.862265978300663e-54),
 np.float64(8.517885103052895e-51),
 np.float64(1.002227760597784e-50),
 np.float64(2.8080489776012833e-50),
 np.float64(1.6168152574521297e-48),
 np.float64(3.048755367297252e-45),
 np.float64(3.8562613645439335e-44),
 np.float64(2.749455914630304e-43),
 np.float64(6.251379297576603e-41),
 np.float64(1.2598308340708517e-40),
 np.float64(1.458986077428999e-40),
 np.float64(3

In [10]:
ligand_activities = predictor.predict_ligand_activities(
    geneset,
    background_expressed_genes,
    potential_ligands
)
ligand_activities = sorted(ligand_activities.items(), key=lambda x : x[1]["aupr_corrected"], reverse=True)
ligand_activities[:10]

NameError: name 'geneset' is not defined